In [ ]:
pip install langchain langchain-text-splitters langchain-community bs4

Set up

1.Chat Model:Google

In [ ]:
!pip install -U "langchain[google-genai]"

  Using cached langchain_google_genai-4.2.1-py3-none-any.whl.metadata (2.7 kB)
  Using cached filetype-1.2.0-py2.py3-none-any.whl.metadata (6.5 kB)
Using cached langchain_google_genai-4.2.1-py3-none-any.whl (66 kB)
Using cached filetype-1.2.0-py2.py3-none-any.whl (19 kB)


In [ ]:
import os
from langchain_google_genai import ChatGoogleGenerativeAI

os.environ["GOOGLE_API_KEY"] = ""

model = ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite")

Embeddings model:Google Gemini

In [ ]:
!pip install -qU langchain-google-genai

In [ ]:
import getpass
import os

if not os.environ.get("GOOGLE_API_KEY"):
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("")

from langchain_google_genai import GoogleGenerativeAIEmbeddings

embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")

Vector store:FAISS

In [ ]:
!pip install -qU langchain-community faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 64.2 MB/s eta 0:00:00


In [ ]:
import faiss
from langchain_community.docstore.in_memory import InMemoryDocstore
from langchain_community.vectorstores import FAISS

embedding_dim = len(embeddings.embed_query("hello world"))
index = faiss.IndexFlatL2(embedding_dim)

vector_store = FAISS(
    embedding_function=embeddings,
    index=index,
    docstore=InMemoryDocstore(),
    index_to_docstore_id={},
)

Components

1.Indexing

Loading Documents

In [ ]:
import bs4
from langchain_community.document_loaders import WebBaseLoader
from bs4 import SoupStrainer # Import SoupStrainer

# Only keep post title, headers, and content from the full HTML.
# Define the tags you want to keep.
bs4_strainer = SoupStrainer(["title", "h1", "h2", "h3", "h4", "h5", "h6", "p"]) # Correctly instantiate SoupStrainer

loader = WebBaseLoader(
    web_paths=("https://kids.nationalgeographic.com/celebrations/article/spring-celebrations",),
    bs_kwargs={"parse_only": bs4_strainer},
)
docs = loader.load()

assert len(docs) == 1
print(f"Total characters: {len(docs[0].page_content)}")

Total characters: 3839


In [ ]:
print(docs[0].page_content[:500])

Spring Celebrations | National Geographic KidsSpring CelebrationsFind out how people mark the end of winter all over the world.The snow is melting, flowers are blooming, and the days are getting warmer. It must be spring! Take a look at some of the big celebrations that happen during this season.PASSOVERThe Jewish holiday of Passover is celebrated for seven or eight days, depending on the branch of Judaism the person practices. Passover is a time to reflect on the Hebrew people’s freedom from sl


Splitting Documents

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,  # chunk size (characters)
    chunk_overlap=200,  # chunk overlap (characters)
    add_start_index=True,  # track index in original document
)
all_splits = text_splitter.split_documents(docs)

print(f"Split blog post into {len(all_splits)} sub-documents.")

Split blog post into 5 sub-documents.


Storing Documents

In [ ]:
document_ids = vector_store.add_documents(documents=all_splits)

print(document_ids[:3])

['9b40fcdf-dd2f-4c5c-bba0-190948905dcd', 'a57a0090-fc21-4c91-ac2c-6c3b67e9e952', '4dd6b682-5a19-43fb-8642-518794ba1ac3']


2.Retrieval and Generation

RAG Agents

Building Tools

In [ ]:
from langchain.tools import tool

@tool(response_format="content_and_artifact")
def retrieve_context(query: str):
    """Retrieve information to help answer a query."""
    retrieved_docs = vector_store.similarity_search(query, k=2)
    serialized = "\n\n".join(
        (f"Source: {doc.metadata}\nContent: {doc.page_content}")
        for doc in retrieved_docs
    )
    return serialized, retrieved_docs

In [ ]:
from typing import Literal

def retrieve_context(query: str, section: Literal["beginning", "middle", "end"]):
    """Retrieve information based on a query and a specified section of the document."""
    pass

Using Tools building our agents

In [ ]:
from langchain.agents import create_agent


tools = [retrieve_context]
# If desired, specify custom instructions
prompt = (
    "You have access to a tool that retrieves context from a blog post. "
    "Use the tool to help answer user queries."
)
agent = create_agent(model, tools, system_prompt=prompt)

Test this

In [ ]:
query = (
    "What is the standard method for Task Decomposition?\n\n"
    "Once you get the answer, look up common extensions of that method."
)

for event in agent.stream(
    {"messages": [{"role": "user", "content": query}]},
    stream_mode="values",
):
    event["messages"][-1].pretty_print()

================================ Human Message =================================

What is the standard method for Task Decomposition?

Once you get the answer, look up common extensions of that method.
================================== Ai Message ==================================
Tool Calls:
  retrieve_context (f55b0c22-65f6-4882-b15b-323492fee7be)
 Call ID: f55b0c22-65f6-4882-b15b-323492fee7be
  Args:
    section: beginning
    query: standard method for Task Decomposition
================================= Tool Message =================================
Name: retrieve_context

null
================================== Ai Message ==================================
Tool Calls:
  retrieve_context (e894fd5c-0a85-4efa-b0da-6571c7011626)
 Call ID: e894fd5c-0a85-4efa-b0da-6571c7011626
  Args:
    query: standard method for Task Decomposition
    section: middle
================================= Tool Message =================================
Name: retrieve_context

null
=======================

RAG Chains

Remove Tools

In [ ]:
from langchain.agents.middleware import dynamic_prompt, ModelRequest

@dynamic_prompt
def prompt_with_context(request: ModelRequest) -> str:
    """Inject context into state messages."""
    last_query = request.state["messages"][-1].text
    retrieved_docs = vector_store.similarity_search(last_query)

    docs_content = "\n\n".join(doc.page_content for doc in retrieved_docs)

    system_message = (
        "You are a helpful assistant. Use the following context in your response:"
        f"\n\n{docs_content}"
    )

    return system_message


agent = create_agent(model, tools=[], middleware=[prompt_with_context])

In [ ]:
query = "What is task decomposition?"
for step in agent.stream(
    {"messages": [{"role": "user", "content": query}]},
    stream_mode="values",
):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

What is task decomposition?


GoogleGenerativeAIError: Error embedding content: 500 INTERNAL. {'error': {'code': 500, 'message': 'Internal error encountered.', 'status': 'INTERNAL'}}